In [7]:
!pip -q install pymilvus sentence-transformers llama-index

In [8]:
import sys
from pathlib import Path

project_root = Path('/work')
sys.path.append(str(project_root))

import re
import unicodedata
import random
import time
import json
import numpy as np
from pymilvus import Collection, connections
from sentence_transformers import SentenceTransformer
from src.utils.data_utils import load_documents, select_random_documents
from contextlib import contextmanager

# Import the evaluation functions
from src.utils.evaluation import (
    perform_analysis,
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results,
    calculate_prefix_success_at_k,
    perform_prefix_match_analysis,
    print_prefix_match_analysis
)

# ---------------------------
# Timing context manager
# ---------------------------
@contextmanager
def timer(name):
    omitted_names = {
        "Selecting a random document for sanity check",
        "Encoding query",
        "Searching in Milvus",
        "Processing search results"
    }
    start = time.time()
    yield
    end = time.time()
    if name not in omitted_names and not name.startswith("Retrieval for document"):
        print(f"[Timing] {name}: {end - start:.4f} seconds")

# ---------------------------
# Sanitize collection name
# ---------------------------

def sanitize_collection_name(name: str) -> str:
    # Normalize string to NFKD (decomposes accents)
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if unicodedata.category(c) != 'Mn')  # Remove diacritics
    
    # Replace invalid characters with underscores (only allow letters, numbers, and underscores)
    name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
    
    return name

# ---------------------------
# MilvusSearcher class
# ---------------------------
class MilvusSearcher:
    def __init__(self, collection_name, model_name):
        """Initialize with collection name and model.
           Loading the model is done once."""
        self.collection_name = collection_name
        with timer("Loading SentenceTransformer model"):
            self.model = SentenceTransformer(model_name)
        self.collection = None
        self.is_connected = False
        
    def connect(self):
        """Establish connection to Milvus."""
        with timer("Connecting to Milvus and loading collection"):
            if not self.is_connected:
                connections.connect(
                    alias="default", 
                    host="host.docker.internal", 
                    port="19530"
                )
                self.is_connected = True
                
            if self.collection is None:
                self.collection = Collection(self.collection_name)
                self.collection.load()
    
    def perform_retrieval(self, query, top_k=10):
        """Perform similarity search."""
        with timer("Encoding query"):
            # Compute and normalize query embedding
            query_embedding = self.model.encode([query], convert_to_numpy=True)[0]
            norm = np.linalg.norm(query_embedding)
            if norm > 0:
                query_embedding = query_embedding / norm
        
        # Search parameters
        search_params = {
            "metric_type": "IP",
            "params": {"ef": 50}
        }
        
        with timer("Searching in Milvus"):
            results = self.collection.search(
                data=[query_embedding],
                anns_field="vector",
                param=search_params,
                limit=top_k,
                output_fields=["item_key", "text"]
            )
        
        with timer("Processing search results"):
            output = []
            for hits in results:
                for hit in hits:
                    score = (hit.score + 1) / 2
                    output.append({
                        'item_key': hit.entity.get("item_key"),
                        'text': hit.entity.get("text"),
                        'score': score
                    })
        return output

    def search_by_item_key(self, item_key):
        """Search for records by item_key."""
        if not self.is_connected or self.collection is None:
            self.connect()
        
        results = self.collection.query(
            expr=f"item_key == '{item_key}'",
            output_fields=["item_key", "text", "vector"]
        )
        
        return results
    
    def cleanup(self):
        """Clean up resources."""
        if self.collection:
            try:
                self.collection.release()
            except Exception as e:
                print(f"Error releasing collection for {self.collection_name}: {e}")
        
        if self.is_connected:
            try:
                connections.disconnect("default")
            except Exception as e:
                print(f"Error disconnecting for {self.collection_name}: {e}")

# ---------------------------
# Main pipeline function
# ---------------------------
def main(file):
    overall_start = time.time()
    searcher = None
    searcher_texto = None
    
    # Model options
    model_options = {
        "hiiamsid": "hiiamsid/sentence_similarity_spanish_es",
        "somosnlp": "somosnlp-hackathon-2022/paraphrase-spanish-distilroberta",
        "mrm8488": "mrm8488/multilingual-e5-large-ft-sts-spanish-matryoshka-768-16-5e"
    }
    
    # Setup paths and names
    documents_path_texto = Path('/work/data/processed/' + file + '_texto.pkl')
    collection_name_texto = file.replace(" ","_") + '_texto_collection'
    collection_name_texto = sanitize_collection_name(collection_name_texto)
    
    documents_path_resumen = Path('/work/data/processed/' + file + '_resumen.pkl')
    collection_name_resumen = file.replace(" ", "_") + '_resumen_collection'
    collection_name_resumen = sanitize_collection_name(collection_name_resumen)
    
    chosen_model = model_options["hiiamsid"]
    
    try:
        # Initialize searchers and connect to Milvus
        with timer("Initializing MilvusSearcher and connecting"):
            searcher = MilvusSearcher(collection_name_resumen, chosen_model)
            searcher.connect()
        with timer("Initializing MilvusSearcher for _texto collection and connecting"):
            searcher_texto = MilvusSearcher(collection_name_texto, chosen_model)
            searcher_texto.connect()
        
        # Load documents
        with timer("Loading documents"):
            documents_resumen = load_documents(documents_path_resumen)
            if documents_resumen is None:
                print("Failed to load documents. Exiting.")
                return
        
        # Select random documents
        sample = min(len(documents_resumen), 16590) # 16590 is the sample size for an infinite population with 99% confidence and 0.01 margin of error
        with timer("Selecting random documents"):
            random_resumen = select_random_documents(documents_resumen, sample)
        
        # ---------------------------
        # Sanity Check
        # ---------------------------
        with timer("Selecting a random document for sanity check"):
            random_doc = random.choice(random_resumen)
            random_doc_key, random_doc_text = random_doc
            print("❤️‍🩹 Selected random document for sanity check:")
            print(f"Item key: {random_doc_key}")
            print(f"Text: {random_doc_text}\n")
    
        with timer("Performing retrieval in sanity check"):
            sanity_results = searcher.perform_retrieval(random_doc_text, top_k=5)
        print("Sanity Retrieval Results:")
        for i, res in enumerate(sanity_results, 1):
            print(f"\nResult {i}:")
            print(f"Item key: {res['item_key']}")
            print(f"Text: {res['text']}")
            print(f"Score: {res['score']:.4f}")
        
        # ---------------------------
        # Additional Retrieval for Evaluation
        # ---------------------------
        with timer("🗒️ Performing retrieval for all selected resumen documents"):
            all_results = []
            for i, (item_key, text) in enumerate(random_resumen, 1):
                with timer(f"Retrieval for document {i}"):
                    results = searcher_texto.perform_retrieval(text, top_k=10)
            
                    # Fetch records safely
                    target_texto_results = searcher_texto.search_by_item_key(item_key)
                    resumen_vector_results = searcher.search_by_item_key(item_key)

                    # Ensure results are not empty
                    if not target_texto_results or not resumen_vector_results:
                        print(f"Warning: No results found for item_key: {item_key}")
                        continue  # Skip this iteration if no results
            
                    # Assume first result is the correct one (modify logic if necessary)
                    target_texto = target_texto_results[0]  # First dictionary from list
                    resumen_vector = resumen_vector_results[0]  # First dictionary from list
            
                    # Ensure 'vector' field exists
                    if "vector" not in target_texto or "vector" not in resumen_vector:
                        print(f"Warning: Missing 'vector' field for item_key: {item_key}")
                        continue  # Skip this iteration if 'vector' field is missing
            
                    # Convert vectors to NumPy arrays
                    target_vector = np.array(target_texto["vector"])
                    resumen_vector = np.array(resumen_vector["vector"])
            
                    # Compute cosine similarity
                    target_score = np.dot(resumen_vector, target_vector)
                    target_score = (target_score + 1) / 2  # Milvus scaling       
            
            
                all_results.append({
                    'resumen_item_key': item_key,
                    'resumen_text': text,
                    'target_text': target_texto['text'],
                    'target_score': target_score,
                    'texto_results': results
                })                
        
        # Perform overall analysis
        with timer("Performing analysis on retrieval results"):
            analysis_results = perform_analysis(all_results)
        
        print("\nAnalysis Results:")
        for metric, value in analysis_results.items():
            print(f"{metric}: {value:.4f}")
        
        # Perform single-match analysis
        with timer("Performing single-match analysis"):
            single_match_metrics = perform_single_match_analysis(all_results, top_k=10)
            print_single_match_analysis(single_match_metrics, top_k=10)

        # Perform prefix-matching analysis
        with timer("Performing prefix-matching analysis"):
            prefix_length = 6  # Match first 6 characters of item_key
            prefix_match_metrics = perform_prefix_match_analysis(
                all_results, 
                top_k=10, 
                prefix_length=prefix_length
            )
            print_prefix_match_analysis(
                prefix_match_metrics, 
                top_k=10, 
                prefix_length=prefix_length
            )
        
        # Print sample results
        with timer("Printing sample results"):
            print_sample_results(all_results, num_samples=5, top_k=5)
    
    except Exception as e:
        print(f"An error occurred: {str(e)}")
    
    finally:
        with timer("Cleaning up resources"):
            if searcher is not None:
                searcher.cleanup()
            if searcher_texto is not None:
                searcher_texto.cleanup()
        overall_end = time.time()
        print(f"\nTotal execution time: {overall_end - overall_start:.2f} seconds")



In [9]:
main('OEB')

[Timing] Loading SentenceTransformer model: 0.8335 seconds
[Timing] Connecting to Milvus and loading collection: 4.1468 seconds
[Timing] Initializing MilvusSearcher and connecting: 4.9804 seconds
[Timing] Loading SentenceTransformer model: 0.6241 seconds
[Timing] Connecting to Milvus and loading collection: 4.5303 seconds
[Timing] Initializing MilvusSearcher for _texto collection and connecting: 5.1545 seconds
[Timing] Loading documents: 1.8517 seconds
[Timing] Selecting random documents: 0.0328 seconds
❤️‍🩹 Selected random document for sanity check:
Item key: OEB170dbebb
Text: Canalización metálica superficial con  4  tubos de acero galvanizado de 2'' de diámetro. (Cualquier franja horaria/3 <== i < "5" horas/Volumen escaso)

[Timing] Performing retrieval in sanity check: 0.2935 seconds
Sanity Retrieval Results:

Result 1:
Item key: OEB170dbebb
Text: Canalización metálica superficial con  4  tubos de acero galvanizado de 2'' de diámetro. (Cualquier franja horaria/3 <== i < "5" horas/V